# 01. Numbers and strings

**Tier:** Deep. **Why you'd use this:** Money and measurements go wrong when binary floats look exact, and string methods always return new strings. You pick `Decimal` or `Fraction` when base 10 must be exact.

**After this lesson you can:** explain `0.1 + 0.2`; choose `Decimal` or `Fraction` when base 10 must be exact; explain why a string method returns a new string.
About 30 minutes. **Uses:** [Numbers](../../course/01-basics/02-numbers-lesson.ipynb), [Strings](../../course/02-strings/01-indexing-lesson.ipynb), [How dict and set work](../04-dicts/01-hash-table-lesson.ipynb)


## Quick review

1. Why must dict keys be hashable?
2. What order do modern dicts iterate in?

<details><summary>Answers</summary>

Lookup finds a slot from the hash; a changing hash would miss. Insertion order.
</details>


## The idea

An `int` grows to as many digits as you need. There is no fixed width and no silent wrap. A `float` is a fixed-size binary fraction. Most decimal fractions, including `0.1`, are not exact in binary, so arithmetic that looks like tenths is really arithmetic on the nearest binary values.

A `str` is a sequence of Unicode code points, and it is immutable. Methods such as `replace` and `upper` allocate a new string. The original object stays as it was, which is why a string can be a dict key and a list cannot.

<details><summary>Explain more</summary>

`float.hex` shows the binary fraction the float actually holds. `Decimal` with a string argument keeps the base-10 digits you wrote. `Fraction` keeps a numerator and a denominator. Both are exact for the values they are given. A float that you then convert has already been rounded.

CPython detail: small integers, and some strings, may be one shared object. `is` can then be true for equal values. That cache is not a promise. Use `==`.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

Python ints are closer to `BigInteger` than to a 32-bit `int`. Floats are IEEE-754 binary64, the same idea as `double`. Strings are immutable, as in Java.
</details>


## See it work

A receipt adds prices that were typed as tenths. Adding them as floats drifts. Parsing the text with `Decimal`, or building a `Fraction` from the cents, keeps the total you would get by hand.


In [1]:
from decimal import Decimal
from fractions import Fraction


def total_cents(prices: list[str]) -> Decimal:
    # 1. Parse the text. Do not start from a float.
    amount = Decimal(0)
    for price in prices:
        amount += Decimal(price)
    return amount


def total_fraction(prices: list[str]) -> Fraction:
    amount = Fraction(0)
    for price in prices:
        amount += Fraction(price)
    return amount


prices = ["0.1", "0.2"]
print(total_cents(prices) == Decimal("0.3"))
print(total_fraction(prices) == Fraction("3/10"))
print(0.1 + 0.2 == 0.3)

True
True
False


| values | float `== 0.3` | Decimal | Fraction |
| --- | --- | --- | --- |
| 0.1 and 0.2 | False | 0.3 | 3/10 |

```
0.1 decimal --> nearest binary64 --> add --> not the nearest binary64 of 0.3
"0.1" -------> Decimal digits -----> add --> exactly 0.3
```


## Check yourself

Answer these from the example above before you run the cell.


1. Why does `2**100` stay positive and exact?
2. Why does `name.replace(...)` leave `name` unchanged?
3. Why is `Decimal(0.1)` already wrong before the addition?

<details><summary>Answers</summary>

1. The integer grows extra digits instead of wrapping at a fixed width. The result is exactly two to the hundredth, not a rounded stand-in.
2. Strings have no operation that writes into an existing object. `replace` allocates a new one. The name `name` still points at the original.
3. `0.1` as a float is already the nearest binary value, not one tenth. Decimal copies that value. It cannot recover digits the float never stored.
</details>


In [2]:
print(0.1 + 0.2 == 0.3)
print(2**100)
name = "ada"
print(name.replace("a", "o"))
print(name)

False
1267650600228229401496703205376
odo
ada


<details><summary>Check your answers</summary>

- `0.1 + 0.2 == 0.3` is `False`. The sum of the two binary approximations is not the binary approximation of `0.3`.
- `2**100` is a 31-digit integer, still an `int`. Nothing wrapped around.
- `replace` prints `odo`. `name` is still `ada`. The method returned a new string.
</details>


## Why it works this way

An `int` is a sign plus an array of digits in a large base. More digits mean a longer array. `2**100` allocates as many digits as the power needs, so there is no maximum like 2³¹−1 and no wrap to a negative number. Arithmetic walks those digits. Adding two n-digit numbers takes time proportional to n. That cost is why a tight loop of huge integers is slower than the same loop on small ones, and why the unlimited range is not free. The design reason is that silent wrap is a worse bug than a slower big integer. Cryptography and exact counts want the extra digits. A fixed-width type would be a different object, and Python does not make you pick one to write `2**100`.

A `float` is a sign, a power of two, and a fixed number of bits of fraction. `0.1` is not a sum of powers of one-half, so the stored value is the closest value that is. `float.hex` prints that value as a hex fraction times a power of two. Adding two such approximations can land slightly off the approximation of the decimal result you expected, which is why `==` against `0.3` fails. The error is tiny next to the magnitude, and it is real. Rounding for display hides it. Comparing with `==` does not.

`Decimal("0.1")` stores base-10 digits, so the addition you do on paper is the addition it does. Constructing `Decimal(0.1)` from a float stores the float's already-rounded binary value, and the damage is done. `Fraction` stores an exact ratio. `Fraction("0.1")` is 1/10. `Fraction(0.1)` is the float's ratio, a long numerator over a long denominator. Pass strings, or integers, when you want the decimal you typed.

A string's code points cannot be replaced in place. `replace` allocates a new string and copies the pieces it keeps. The original object's identity is unchanged, so every name that pointed at it still sees the old text. That is why strings are hashable: the hash cannot be invalidated by a later write, because a write is not possible. Building a long string with `+=` in a loop allocates a new string each time and copies a longer prefix, which is quadratic. `"".join` allocates once. The cost of immutability is those copies. The benefit is safe sharing and legal dict keys.

CPython detail: integers in a small range, and some strings that look like identifiers, may be interned, one object for many names. `(1 + 1) is 2` can be true. Two separately built long strings of the same text can be two objects. The cell is volatile because other builds do not promise the cache. `==` is the comparison that matches the meaning.

The evidence cells print the failed float equality, the hex fraction, the exact decimal and rational totals, and a string whose `replace` does not change the original.


In [3]:
print(0.1 + 0.2 == 0.3)
print((0.1).hex())
print((0.1 + 0.2).hex())
print((0.3).hex())

False
0x1.999999999999ap-4
0x1.3333333333334p-2
0x1.3333333333333p-2


In [4]:
name = "ada"
updated = name.replace("a", "o")
print(updated, name, updated is name)

odo ada False


In [5]:
# CPython detail: small ints and some strings may be one shared object.
two = 2
print((1 + 1) is two)
left = "ada"
right = "ada"
print(left is right)

True
True


## When to use it

Use `int` for counts, indexes, and anything that must be exact and whole. Use `float` for scientific magnitudes where a tiny relative error is acceptable and you want speed. Use `Decimal` for money and for any total a person typed in base 10. Use `Fraction` when the exact ratio matters and the inputs are ratios or integers.

Do not start from a float and then ask `Decimal` or `Fraction` to recover the decimal you meant. Pass the original text. Do not use `is` on numbers or strings. Do not build a long string with `+=` in a loop. Join a list of pieces.

A float that fails `==` can still be close enough. Compare with a tolerance when the inputs were floats and the specification allows error. When the specification is "the cents add up", tolerance is the wrong tool. Exact decimal is the tool.


## Common mistake

`Decimal(0.1)` looks like "use Decimal for tenths". The argument is already a float, so Decimal stores the binary approximation and the sum still is not `0.3`. Nothing raises. The total is a Decimal, which makes the bug look fixed.


In [6]:
from decimal import Decimal

print(Decimal(0.1) + Decimal(0.2) == Decimal("0.3"))
print(Decimal(0.1))

False
0.1000000000000000055511151231257827021181583404541015625


The comparison is `False`. `Decimal(0.1)` printed a long coefficient, the float's value written in base 10, not the two characters `0.1`. The fix parses the text you actually meant.


In [7]:
from decimal import Decimal

print(Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))

True


## Try it

Change `total_cents` to add `Decimal(float(price))` instead of `Decimal(price)`. Why does the total stop equaling `Decimal("0.3")`, and which conversion threw the decimal digits away?


## What this explains

- [Numbers](../../course/01-basics/02-numbers-lesson.ipynb): `int` has no fixed width, and `float` is a binary approximation. `//` and the other operators sit on top of these representations.
- [Floats](../../course/10-pitfalls/05-floats-lesson.ipynb): `0.1 + 0.2` is the same binary fact, and `Decimal` is the repair when you still have the original text.
- [Indexing](../../course/02-strings/01-indexing-lesson.ipynb): a string method returns a new string because the code points cannot be overwritten.
- [How dict and set work](../04-dicts/01-hash-table-lesson.ipynb): immutability is why a string is a legal key and a list is not.


## Recap

- `int` is exact and grows. `float` is a fixed binary fraction. `0.1` is not exact.
- `Decimal` and `Fraction` stay exact when you give them text or integers, not a float that already rounded.
- Strings are immutable, so methods return new objects, and the text can be a dict key.

Next: [Scopes and closures](../06-scopes/01-closures-lesson.ipynb).
